# Tutorial 11: PhysiCell Integration

**Duration:** 20-25 minutes

This tutorial covers how to analyze agent-based model (ABM) outputs from PhysiCell using spatialtissuepy.

## Learning Objectives

By the end of this tutorial, you will be able to:
- Load PhysiCell simulation outputs
- Extract spatial data from individual timesteps
- Track spatial metrics over simulation time
- Compare multiple simulation runs (parameter sweeps)
- Use spatial analysis to validate ABM predictions

## Prerequisites

- Tutorials 1-10 completed
- Basic understanding of agent-based modeling
- (Optional) PhysiCell installation for running your own simulations

## Biological Context

**Why analyze ABM outputs?**
- Validate simulation predictions against experimental data
- Track how spatial organization evolves over time
- Compare parameter regimes (e.g., drug concentrations)
- Generate hypotheses for experimental testing

**PhysiCell:**
- Open-source physics-based cell simulator
- Outputs MultiCellDS XML format
- Commonly used for tumor growth, immune interactions

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from spatialtissuepy import SpatialTissueData
from spatialtissuepy.synthetic import (
    PhysiCellSimulation,
    PhysiCellExperiment,
    PhysiCellTimeStep,
    read_physicell_timestep,
)
from spatialtissuepy.summary import StatisticsPanel, SpatialSummary
from spatialtissuepy.viz import plot_spatial_scatter, plot_trajectory

np.random.seed(42)

## Section 1: Understanding PhysiCell Output Structure

PhysiCell outputs data in the MultiCellDS format:
```
output/
├── initial.xml          # Initial conditions
├── output00000000.xml   # Timestep 0
├── output00000001.xml   # Timestep 1
├── ...
├── output00000100.xml   # Timestep 100
└── PhysiCell_settings.xml  # Simulation parameters
```

Each XML file contains:
- Cell positions (x, y, z)
- Cell types/phenotypes
- Cell states (live, dead, cycling, etc.)
- Custom variables

## Section 2: Simulating PhysiCell-like Data

Since you may not have PhysiCell outputs available, we'll create synthetic data that mimics PhysiCell output structure.

In [ ]:
def simulate_tumor_growth(n_timesteps=50, dt=1.0):
    """
    Simulate tumor growth dynamics (simplified).
    
    Returns list of SpatialTissueData objects representing timesteps.
    """
    timesteps = []
    
    # Initial tumor cells
    n_tumor = 50
    tumor_center = np.array([500, 500])
    tumor_coords = np.random.normal(tumor_center, 50, (n_tumor, 2))
    
    # Initial immune cells (surrounding)
    n_immune = 30
    immune_coords = np.random.uniform(0, 1000, (n_immune, 2))
    
    for t in range(n_timesteps):
        # Tumor growth (exponential with carrying capacity)
        growth_rate = 0.05 * (1 - n_tumor / 500)
        new_tumor = int(n_tumor * growth_rate)
        
        if new_tumor > 0:
            # New cells near existing ones
            parent_idx = np.random.choice(len(tumor_coords), new_tumor)
            new_coords = tumor_coords[parent_idx] + np.random.normal(0, 20, (new_tumor, 2))
            tumor_coords = np.vstack([tumor_coords, new_coords])
            n_tumor = len(tumor_coords)
        
        # Immune cell dynamics (attracted to tumor)
        tumor_centroid = tumor_coords.mean(axis=0)
        attraction = 0.1 * (tumor_centroid - immune_coords)
        immune_coords = immune_coords + attraction + np.random.normal(0, 5, immune_coords.shape)
        
        # Add new immune cells occasionally
        if t % 5 == 0 and len(immune_coords) < 100:
            new_immune = np.random.uniform(0, 1000, (5, 2))
            immune_coords = np.vstack([immune_coords, new_immune])
        
        # Create timestep
        coords = np.vstack([tumor_coords, immune_coords])
        types = np.array(['Tumor'] * len(tumor_coords) + ['Immune'] * len(immune_coords))
        
        tissue = SpatialTissueData(coordinates=coords, cell_types=types)
        timesteps.append({'time': t * dt, 'data': tissue})
    
    return timesteps

# Simulate
simulation_data = simulate_tumor_growth(n_timesteps=50)
print(f"Simulated {len(simulation_data)} timesteps")
print(f"\nTimestep 0: {simulation_data[0]['data'].n_cells} cells")
print(f"Timestep 49: {simulation_data[-1]['data'].n_cells} cells")

In [ ]:
# Visualize evolution
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

timesteps_to_show = [0, 10, 20, 30, 40, 49]

for ax, t in zip(axes.flat, timesteps_to_show):
    tissue = simulation_data[t]['data']
    plot_spatial_scatter(tissue, ax=ax)
    ax.set_title(f'Time = {simulation_data[t]["time"]:.0f}\n({tissue.n_cells} cells)')

plt.tight_layout()
plt.show()

## Section 3: Tracking Metrics Over Time

### 3.1 Cell Count Trajectories

In [ ]:
# Track cell counts over time
times = []
tumor_counts = []
immune_counts = []

for ts in simulation_data:
    tissue = ts['data']
    times.append(ts['time'])
    tumor_counts.append(np.sum(tissue.cell_types == 'Tumor'))
    immune_counts.append(np.sum(tissue.cell_types == 'Immune'))

# Plot trajectories
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(times, tumor_counts, 'r-', linewidth=2, label='Tumor')
ax.plot(times, immune_counts, 'b-', linewidth=2, label='Immune')
ax.set_xlabel('Time')
ax.set_ylabel('Cell Count')
ax.set_title('Cell Population Dynamics')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 3.2 Spatial Metric Trajectories

In [ ]:
# Define metrics panel
panel = StatisticsPanel(name='abm_tracking')
panel.add('cell_counts')
panel.add('mean_nearest_neighbor_distance')
panel.add('mean_neighborhood_entropy', radius=50.0)
panel.add('colocalization_score', type_a='Tumor', type_b='Immune', radius=50.0)

# Compute metrics for each timestep
metrics_over_time = []

for ts in simulation_data[::5]:  # Every 5th timestep for speed
    tissue = ts['data']
    summary = SpatialSummary(tissue, panel)
    results = summary.to_dict()
    results['time'] = ts['time']
    metrics_over_time.append(results)

metrics_df = pd.DataFrame(metrics_over_time)
print("Tracked metrics:")
print(metrics_df.head())

In [ ]:
# Plot spatial metrics over time
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Tumor-Immune colocalization
coloc_col = [c for c in metrics_df.columns if 'coloc' in c][0] if any('coloc' in c for c in metrics_df.columns) else None

axes[0, 0].plot(metrics_df['time'], metrics_df['n_Tumor'], 'r-o')
axes[0, 0].set_xlabel('Time')
axes[0, 0].set_ylabel('Tumor Cell Count')
axes[0, 0].set_title('Tumor Growth')

axes[0, 1].plot(metrics_df['time'], metrics_df['n_Immune'], 'b-o')
axes[0, 1].set_xlabel('Time')
axes[0, 1].set_ylabel('Immune Cell Count')
axes[0, 1].set_title('Immune Recruitment')

if 'mean_nn_distance' in metrics_df.columns:
    axes[1, 0].plot(metrics_df['time'], metrics_df['mean_nn_distance'], 'g-o')
    axes[1, 0].set_ylabel('Mean NN Distance')
axes[1, 0].set_xlabel('Time')
axes[1, 0].set_title('Cell Density (inverse of NN distance)')

if coloc_col:
    axes[1, 1].plot(metrics_df['time'], metrics_df[coloc_col], 'm-o')
    axes[1, 1].set_ylabel('Colocalization Score')
axes[1, 1].set_xlabel('Time')
axes[1, 1].set_title('Tumor-Immune Colocalization')

plt.tight_layout()
plt.show()

## Section 4: Comparing Simulations (Parameter Sweeps)

### 4.1 Simulating Different Conditions

In [ ]:
def simulate_with_treatment(treatment_strength=0.0, n_timesteps=50):
    """
    Simulate tumor growth with treatment effect.
    Higher treatment_strength = more tumor cell death.
    """
    timesteps = []
    
    n_tumor = 50
    tumor_center = np.array([500, 500])
    tumor_coords = np.random.normal(tumor_center, 50, (n_tumor, 2))
    
    n_immune = 30
    immune_coords = np.random.uniform(0, 1000, (n_immune, 2))
    
    for t in range(n_timesteps):
        # Growth (reduced by treatment)
        growth_rate = 0.05 * (1 - treatment_strength) * (1 - n_tumor / 500)
        
        # Treatment-induced death
        death_rate = 0.02 * treatment_strength
        n_deaths = int(n_tumor * death_rate)
        if n_deaths > 0 and len(tumor_coords) > n_deaths:
            keep_idx = np.random.choice(len(tumor_coords), len(tumor_coords) - n_deaths, replace=False)
            tumor_coords = tumor_coords[keep_idx]
        
        # Growth
        n_tumor = len(tumor_coords)
        new_tumor = int(max(0, n_tumor * growth_rate))
        
        if new_tumor > 0 and len(tumor_coords) > 0:
            parent_idx = np.random.choice(len(tumor_coords), new_tumor)
            new_coords = tumor_coords[parent_idx] + np.random.normal(0, 20, (new_tumor, 2))
            tumor_coords = np.vstack([tumor_coords, new_coords])
            n_tumor = len(tumor_coords)
        
        # Immune dynamics
        if len(tumor_coords) > 0:
            tumor_centroid = tumor_coords.mean(axis=0)
            attraction = 0.1 * (tumor_centroid - immune_coords)
            immune_coords = immune_coords + attraction + np.random.normal(0, 5, immune_coords.shape)
        
        if t % 5 == 0 and len(immune_coords) < 100:
            new_immune = np.random.uniform(0, 1000, (5, 2))
            immune_coords = np.vstack([immune_coords, new_immune])
        
        # Create timestep
        if len(tumor_coords) > 0:
            coords = np.vstack([tumor_coords, immune_coords])
            types = np.array(['Tumor'] * len(tumor_coords) + ['Immune'] * len(immune_coords))
        else:
            coords = immune_coords
            types = np.array(['Immune'] * len(immune_coords))
        
        tissue = SpatialTissueData(coordinates=coords, cell_types=types)
        timesteps.append({'time': t, 'data': tissue})
    
    return timesteps

# Simulate different treatment conditions
conditions = {
    'Control': simulate_with_treatment(0.0),
    'Low Dose': simulate_with_treatment(0.3),
    'High Dose': simulate_with_treatment(0.6),
}

print("Simulated conditions:")
for name, sim in conditions.items():
    final = sim[-1]['data']
    n_tumor = np.sum(final.cell_types == 'Tumor') if 'Tumor' in final.cell_types else 0
    print(f"  {name}: Final tumor count = {n_tumor}")

In [ ]:
# Compare tumor growth trajectories
fig, ax = plt.subplots(figsize=(10, 6))

colors = {'Control': 'red', 'Low Dose': 'orange', 'High Dose': 'green'}

for name, sim in conditions.items():
    times = [ts['time'] for ts in sim]
    tumor_counts = []
    for ts in sim:
        tissue = ts['data']
        n_tumor = np.sum(tissue.cell_types == 'Tumor') if 'Tumor' in tissue.cell_types else 0
        tumor_counts.append(n_tumor)
    
    ax.plot(times, tumor_counts, color=colors[name], linewidth=2, label=name)

ax.set_xlabel('Time')
ax.set_ylabel('Tumor Cell Count')
ax.set_title('Treatment Response: Tumor Growth Trajectories')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 4.2 Spatial Comparison at Endpoint

In [ ]:
# Compare final spatial distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (name, sim) in zip(axes, conditions.items()):
    final_tissue = sim[-1]['data']
    plot_spatial_scatter(final_tissue, ax=ax)
    n_tumor = np.sum(final_tissue.cell_types == 'Tumor') if 'Tumor' in final_tissue.cell_types else 0
    ax.set_title(f'{name}\nFinal: {n_tumor} tumor cells')

plt.tight_layout()
plt.show()

## Section 5: Loading Real PhysiCell Data

When you have actual PhysiCell outputs, you can load them as follows:

```python
# Load single simulation
sim = PhysiCellSimulation.from_output_folder('./output')

# Get available timesteps
print(sim.timesteps)

# Load specific timestep
ts = sim.get_timestep(100)

# Convert to SpatialTissueData
tissue = ts.to_spatial_data()

# Get cell type mapping
print(ts.cell_type_names)

# Compute summary with panel
df = sim.summarize(panel)
```

For multiple simulations (parameter sweep):

```python
# Load experiment
exp = PhysiCellExperiment.from_folders([
    './output_ctrl',
    './output_drug_low',
    './output_drug_high'
])

# Summarize all simulations
master_df = exp.summarize(panel)
```

## Exercise: Analyze Treatment Effects

1. Track the colocalization between Tumor and Immune cells over time for each condition
2. At what timepoint does the High Dose condition show different spatial organization?
3. Compute and compare the immune infiltration score at the final timestep

In [ ]:
# Your code here


## Summary

In this tutorial, you learned:

- **PhysiCell output structure:** MultiCellDS XML format
- **Loading simulations:** Using PhysiCellSimulation and PhysiCellExperiment
- **Temporal tracking:** Following metrics over simulation time
- **Parameter comparison:** Comparing different simulation conditions
- **Validation workflows:** Using spatial analysis to interpret ABM results

**Key insights:**
- ABM outputs are naturally suited for spatial analysis
- Temporal tracking reveals dynamics not captured in snapshots
- Spatial metrics can distinguish treatment effects
- Same analysis pipeline for simulations and experimental data

## Next Steps

- **Tutorial 12: Advanced Workflows** - Complete analysis pipelines